In [5]:
from ollama import Client

model='llama3.1:8b'
client = Client(host='http://localhost:11434')

In [6]:
# Test ollama client
response = client.chat(
    model=model,
    messages=[
        {
            'role': 'user',
            'content': 'Привет! Напиши одно короткое предложение.',
        },
    ],
)
print(response['message']['content'])

Привет! Меня зовут Лукас.


In [7]:
import re
import json

EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 


class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str, product_id_req: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""

    ret = {"product_id": product_id_req}

    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        return ret | {'error': f"Ответ не является корректным JSON: {error.msg}"}

    if not isinstance(data, dict):
        return ret | {'error': "На верхнем уровне должен находиться JSON-объект"}

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        }

    if unexpected_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        }

    product_id = data['product_id']
    if product_id.strip() != product_id_req.strip():
        return ret | {'error': "Поле product_id не соответствует входным параметрам"}

    description = data["description"]

    if not isinstance(description, str):
        return ret | {'error': "Поле description должно быть строкой"}

    description = description.strip()

    if not description:
        return ret | {'error': "Поле description не должно быть пустым"}
      
    i = count_sentences(description)
    if i < 2 or i > 5:
        return ret | {'error': "Description must contain 2-5 sentences"}

    if len(description) < MIN_DESCRIPTION_LENGTH:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Поле description содержит больше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    if len(description) > MAX_DESCRIPTION_LENGTH:
        return ret | {'error': "Поле description содержит меньше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    pros = data['pros']
    if len(pros) < 2 or len(pros) > 5:
       return ret | {'error': "Поле pros must contain 2-5 values"}

    cons = data['cons']
    if len(cons) < 1 or len(cons) > 3:
       return ret | {'error': "Поле cons must contain 1-3 values"}
    
    tags = data['tags']
    if len(tags) < 3 or len(tags) > 8:
       return ret | {'error': "Поле tags must contain 3-8 values"}
    
    return ret

In [8]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)

class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [9]:
response_schema = CardRequest.model_json_schema()
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [10]:
def request_one(
    msg: list[dict], 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> tuple[str, int]:
  messages=[
    {"role": "system", "content": 
      """
        Generate a product card in Russian (utf-8) by example.
        Example:
        {
          "product_id": "P-1001",
          "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
          "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
          "cons": ["Быстро разряжаются на морозе"],
          "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
        } 
        "description": 2-5 sentences,
        return only JSON
      """
    } 
  ] + msg
  # print('>>> messages', messages)

  response = client.chat(
      model=model,  
      messages=messages,
      options={
        'seed':               42,  
        'temperature':        params['temperature'],  
        'top_p':              params['top_p'],
        'max_tokens':         params['max_tokens'],   
        "top_k":              params['top_k'], 
        "repetition_penalty": params['repetition_penalty'],      
      },
      format=CardRequest.model_json_schema(),
  )

  content = response['message']['content']
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )

  output_tokens = response.eval_count
  
  return content, output_tokens

In [11]:
import time

def worming_up():
  item = {"product_id": "P-0026", "title": "Смартфон Grafo G9", "category": "Электроника / Смартфоны", "attributes": {"бренд": "Grafo", "экран": "6.7\"", "память": "256 ГБ", "оперативная память": "8 ГБ", "аккумулятор": "6000 мА·ч", "основная камера": "50 Мп"}, "reviews": ["Пользуюсь пару недель. Камера снимает достойно даже вечером", "Брал для себя. Звонки чистые, собеседника слышно хорошо", "В целом Быстро заряжается, за час почти полный", "Заказывал на подарок. Работает шустро, приложения не тормозят", "Брал для себя. Камера мылит фото при недостатке света", "Пользуюсь пару недель. К вечеру заряд просаживается быстрее ожидаемого", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь каждый день. Экран яркий, на солнце всё видно", "Быстро заряжается, за час почти полный", "В руке лежит удобно, не скользит", "Динамик тихий, в шумном месте плохо слышно", "Работает шустро, приложения не тормозят", "В руке лежит удобно, не скользит", "Коротко: Работает шустро, приложения не тормозят", "Купил на замену старому. Обычный телефон на каждый день, без сюрпризов", "Брал для себя. Батарея держит полтора дня без подзарядки", "В целом Работает шустро, приложения не тормозят"]}

  row_dict = [{"role": "user", "content": f"""{item}"""}]

  started = time.perf_counter()
  resp, tokens = request_one(row_dict)
  generation_time = time.perf_counter() - started

  print('>>> worming_up response, tokens, time:\n', resp, tokens, generation_time)

In [55]:
worming_up()

>>> worming_up response, tokens, time:
 {
  "product_id": "P-0026",
  "description": "Смартфон Grafo G9 - это мощный и функциональный смартфон с большим экраном 6,7 дюймов и мощной основной камерой 50 Мп. Он имеет 256 ГБ внутренней памяти и 8 ГБ оперативной памяти, что позволяет легко хранить и загружать приложения. Аккумулятор имеет емкость 6000 мА·ч, что обеспечивает долгую работу без подзарядки.",
  "pros": ["Чистый звон", "Удобная посадка", "Быстро заряжается", "Работает шустро, приложения не тормозят", "Экран яркий, на солнце всё видно"],
  "cons": ["Динамик тихий, в шумном месте плохо слышно", "Камера мылит фото при недостатке света"],
  "tags": ["Смартфон", "Электроника", "Grafo", "6,7 дюймов", "256 ГБ", "8 ГБ", "6000 мА·ч", "50 Мп"]
} 282 18.439344040991273


In [12]:
MAX_ATTEMPTS = 3 

def generate_one(
    msg: list[dict], 
    product_id: str,
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350},
) -> tuple[str, int]:
  messages_history = msg.copy()

  counter = 0
  errors = 0
  for attempt in range(1, MAX_ATTEMPTS + 1):
    counter += 1
    content, tokens = request_one(messages_history, params)

    error_dict = parse_and_validate(content, product_id)
    if not 'error' in error_dict:
      return content, tokens
    else:
      errors += 1
      print('>>> error: ', error_dict)
      print('>>> error content: ', content)

      if attempt == MAX_ATTEMPTS:
        return {"product_id": product_id, "error": "ResponseValidationError"}

      messages_history = messages_history + [
        {
          "role": "assistant",
          "content": content,
        },
        {
          "role": "user",
          "content": (
              "Предыдущий ответ не прошёл проверку. "
              f"Причина: {error_dict}. "
              "Исправь ответ и верни только JSON"
          ),
        }
      ]

In [13]:
# test generate_one
item = {
  "product_id": "P-0027", 
  "title": "Пылесос Klin Eco", 
  "category": "Бытовая техника - Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, 
  "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", 
                "Насадки удобные, под мебель достаёт легко", 
  ]
}
row_dict = [{"role": "user", "content": f"""{item}"""}]

resp, tokens = generate_one(row_dict, product_id=item['product_id'])
print('>>> resp\n', resp, tokens)

>>> resp
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером объемом 2 литра. Он оснащен щеткой, которая легко чистит поверхности, а также насадками для удаления пыли из труднодоступных мест. Пылесос работает от сети 220 Вт и может работать до 60 минут без подзарядки.",
  "pros": ["Удобные насадки", "Легко чистит поверхности", "Мощность всасывания 220 Вт"],
  "cons": ["Щётка забивается волосами"],
  "tags": ["пылесос", "клин", "беспроводные", "с контейнером", "220 Вт"]
} 187


In [14]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [15]:
import json
from datetime import datetime
from pathlib import Path

def wright_files(timestamp_str_dir, results, params, df):
  timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
  directory = Path(f"./outputs/{timestamp_str_dir}/{timestamp_str}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "results.json", "w", encoding="utf-8") as file:
    file.write(results)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

  totals_path = f"./outputs/{timestamp_str_dir}" / "totals.csv"
  df.to_csv(totals_path, index=False, encoding="utf-8")

In [17]:
import tiktoken

def token_count_tiktoken(text: str) -> int:
  # Llama 3 использует токенизатор, очень похожий на cl100k_base или o200k_base
  encoding = tiktoken.get_encoding("cl100k_base")
  return len(encoding.encode(text))

In [18]:
# test token_count_tiktoken
tokens = token_count_tiktoken('hello world!')
tokens


3

In [16]:
import json

MAX_TOKENS = 1500

def item_normalize(item: str) -> str:
  item_json = json.loads(item)
  reviews = item_json["reviews"][:30]
  # while token_count_ollama('[' + ", ".join(reviews) + ']') > MAX_TOKENS:
  #    reviews.pop() 

  item_json["reviews"] = reviews
  return json.dumps(item_json, ensure_ascii=False)

In [17]:
item = """
{"product_id": "P-0027", "title": "Пылесос Klin Eco", "category": "Бытовая техника / Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", "Насадки удобные, под мебель достаёт легко", "Купил на замену старому. Работает тихо, ребёнка не будит", "Контейнер вытряхивается быстро, руки чистые", "Что могу сказать: Работает тихо, ребёнка не будит", "Пользуюсь каждый день. Насадки удобные, под мебель достаёт легко", "Брал для себя. Лёгкий, руку не устаёт держать", "Насадки удобные, под мебель достаёт легко", "Пользуюсь пару недель. Контейнер вытряхивается быстро, руки чистые", "Брал для себя. Собирает шерсть кота без проблем", "Брали на дачу, для нечастой уборки годится", "Что могу сказать: Работает тихо, ребёнка не будит", "Заказывал на подарок. Работает тихо, ребёнка не будит", "Пользуюсь месяц. Работает тихо, ребёнка не будит", "Купил на замену старому. Насадки удобные, под мебель достаёт легко"]}
"""

item_norm = item_normalize(item)
print('>>> item_norm', item_norm)

>>> item_norm {"product_id": "P-0027", "title": "Пылесос Klin Eco", "category": "Бытовая техника / Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", "Насадки удобные, под мебель достаёт легко", "Купил на замену старому. Работает тихо, ребёнка не будит", "Контейнер вытряхивается быстро, руки чистые", "Что могу сказать: Работает тихо, ребёнка не будит", "Пользуюсь каждый день. Насадки удобные, под мебель достаёт легко", "Брал для себя. Лёгкий, руку не устаёт держать", "Насадки удобные, под мебель достаёт легко", "Пользуюсь пару недель. Контейнер вытряхивается быстро, руки чистые", "Брал для себя. Собирает шерсть кота без проблем", "Брали на дачу, для нечастой уборки годится", "Что могу сказать: Работает тихо, ребёнка не будит", "Заказывал на подарок. Работает тихо, ребёнка не будит", "Пользуюс

In [19]:
import json
from pathlib import Path

def card_save(timestamp_str_dir, card, product_id):
  directory = Path(f"./outputs/{timestamp_str_dir}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(f"{directory}/{product_id}.json", "w", encoding="utf-8") as file:
    file.write(card)


def params_save(timestamp_str_dir, params):
  directory = Path(f"./outputs/{timestamp_str_dir}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

In [24]:
import pandas as pd
from datetime import datetime
import time
import json

items_slice = items[:5]

df = pd.DataFrame(columns=["num", "temperature", "top_p", "top_k", "repetition_penalty", "max_tokens", 'error', 'descr_len', 'tokens', 'time_per_item'])

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

counter = 0

for temperature in [0.0, 0.3, 0.7]:
  for top_p in [0.9, 1.0]:
    for top_k in [20, 50, None]:
      for repetition_penalty in [1.0, 1.1, 1.2]:
        for max_tokens in [250, 350, 500]:
# for temperature in [0.3]:
#   for top_p in [0.9]:
#     for top_k in [50]:
#       for repetition_penalty in [1.1]:
#         for max_tokens in [350]:
          params = {
            "temperature": temperature, 
            "top_p": top_p, 
            "top_k": top_k, 
            "repetition_penalty": repetition_penalty, 
            "max_tokens": max_tokens,
          }
          for item in items_slice:
            err = 0
            counter += 1

            item_norm = item_normalize(item)
            # print('>>> item_norm', item_norm)

            row_dict = {"role": "user", "content": item_norm}
            item_dict = json.loads(item)

            started = time.perf_counter()
            results, tokens = generate_one([row_dict], product_id=item_dict['product_id'], params=params)
            generation_time = time.perf_counter() - started

            if 'error' in results:
              err = 1
              descr_len = 0
            else:
              results_dict = json.loads(results)
              descr_len = len(results_dict['description'])
              # card_save(timestamp_str, results, item_dict['product_id'])
              
            new_row = {'num': counter} | params | {'error': err, 'descr_len': descr_len, 'tokens': tokens, 'time_per_item': round(generation_time, 2)}
            print(new_row)

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            # wright_files(timestamp_str, results, new_row, df)

{'num': 1, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 250, 'error': 0, 'descr_len': 265, 'tokens': 174, 'time_per_item': 8.7}
{'num': 2, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 250, 'error': 0, 'descr_len': 257, 'tokens': 180, 'time_per_item': 9.24}
{'num': 3, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 250, 'error': 0, 'descr_len': 264, 'tokens': 182, 'time_per_item': 9.03}
{'num': 4, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 250, 'error': 0, 'descr_len': 223, 'tokens': 185, 'time_per_item': 9.32}
{'num': 5, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 250, 'error': 0, 'descr_len': 205, 'tokens': 187, 'time_per_item': 9.37}
{'num': 6, 'temperature': 0.0, 'top_p': 0.9, 'top_k': 20, 'repetition_penalty': 1.0, 'max_tokens': 350, 'error': 0, 'descr_len': 265, 'tokens

In [27]:
df_sorted = df.sort_values(by='time_per_item')
df_sorted

,num,temperature,top_p,top_k,repetition_penalty,max_tokens,error,descr_len,tokens,time_per_item
787,788,0.7,1.0,None,1.1,350,0,184,148,7.26
792,793,0.7,1.0,None,1.1,500,0,184,148,7.27
732,733,0.7,1.0,50,1.0,500,0,184,148,7.36
727,728,0.7,1.0,50,1.0,350,0,184,148,7.37
362,363,0.3,0.9,None,1.0,250,0,188,152,7.43
...,...,...,...,...,...,...,...,...,...,...
439,440,0.3,1.0,20,1.2,250,0,215,199,45.21
424,425,0.3,1.0,20,1.1,250,0,215,199,45.26
489,490,0.3,1.0,50,1.2,350,0,215,199,45.27
429,430,0.3,1.0,20,1.1,350,0,215,199,45.47


In [29]:
df_err = df.sort_values(by='error')
df_err

,num,temperature,top_p,top_k,repetition_penalty,max_tokens,error,descr_len,tokens,time_per_item
0,1,0.0,0.9,20,1.0,250,0,265,174,8.7
533,534,0.3,1.0,None,1.2,350,0,212,168,13.88
534,535,0.3,1.0,None,1.2,350,0,215,199,32.45
535,536,0.3,1.0,None,1.2,500,0,241,191,16.87
536,537,0.3,1.0,None,1.2,500,0,196,193,17.47
...,...,...,...,...,...,...,...,...,...,...
274,275,0.3,0.9,20,1.0,250,0,215,199,27.73
275,276,0.3,0.9,20,1.0,350,0,241,191,13.8
276,277,0.3,0.9,20,1.0,350,0,196,193,14.71
266,267,0.0,1.0,None,1.2,500,0,257,180,11.96


In [ ]:
means = (df.groupby(['temperature', 'top_p', 'top_k', 'repetition_penalty', 'max_tokens'], as_index=False)['time_per_item']
        .mean()
        .sort_values(by='time_per_item')
        .to_csv('./outputs/params_mean_sorted.csv', index=False, encoding='utf-8'))

In [44]:
means

temperature  top_p  top_k  repetition_penalty  max_tokens
0.0          0.9    20     1.0                 250            9.132
                                               350           11.266
                                               500           18.984
                           1.1                 250           15.952
                                               350           16.114
                                                              ...  
0.7          1.0    50     1.1                 350           12.288
                                               500           12.248
                           1.2                 250           11.834
                                               350           11.784
                                               500            11.72
Name: time_per_item, Length: 108, dtype: object

In [38]:
# totals_path = Path(f"./outputs/totals_final_{timestamp_str}.csv")
totals_path = Path(f"./outputs/params_sorted.csv")
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")

In [ ]:
totals_path = Path(f"./outputs/totals_final_{timestamp_str}.csv")
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")

In [84]:
# file_name = './dev.jsonl'
file_name = './benchmark.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [93]:
import time
import json

items_slice = items[:]
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
err = 0
count = 0

params = {
  "temperature":        0.3, 
  "top_p":              0.9, 
  "top_k":              50, 
  "repetition_penalty": 1.1, 
  "max_tokens":         350,
  "products":           len(items_slice), 
}

# worming_up()

started_batch = time.perf_counter()

for item in items_slice:
  count += 1

  item_norm = item_normalize(item)
  row_dict = {"role": "user", "content": item_norm}
  item_dict = json.loads(item)

  started = time.perf_counter()
  results, tokens = generate_one([row_dict], product_id=item_dict['product_id'], params=params)
  generation_time = time.perf_counter() - started

  if 'error' in results:
    err += 1
  else:
    card_save(timestamp_str, results, item_dict['product_id'])
    
  new_row = {"num": count, 'card': item_dict['product_id'], 'error': err, 'tokens': tokens, 'time_per_item': round(generation_time, 2)}
  print(new_row)

batch_generation_time = round(time.perf_counter() - started_batch, 2)
params_save(
  timestamp_str, 
  params | 
  {
    "errors": err, 
    "error_percent": round(100 * err / len(items_slice), 2), 
    "total_time_sec": batch_generation_time, 
    'time_per_item_sec': round(batch_generation_time / len(items_slice), 2)
  }
)

{'num': 1, 'card': 'P-1144', 'error': 0, 'tokens': 148, 'time_per_item': 7.46}
{'num': 2, 'card': 'P-1057', 'error': 0, 'tokens': 180, 'time_per_item': 9.01}
{'num': 3, 'card': 'P-1261', 'error': 0, 'tokens': 210, 'time_per_item': 10.33}
{'num': 4, 'card': 'P-1237', 'error': 0, 'tokens': 159, 'time_per_item': 10.31}
{'num': 5, 'card': 'P-1281', 'error': 0, 'tokens': 148, 'time_per_item': 8.58}
{'num': 6, 'card': 'P-1155', 'error': 0, 'tokens': 165, 'time_per_item': 9.3}
{'num': 7, 'card': 'P-1076', 'error': 0, 'tokens': 182, 'time_per_item': 12.25}
{'num': 8, 'card': 'P-1258', 'error': 0, 'tokens': 205, 'time_per_item': 16.35}
{'num': 9, 'card': 'P-1075', 'error': 0, 'tokens': 198, 'time_per_item': 15.62}
{'num': 10, 'card': 'P-1148', 'error': 0, 'tokens': 153, 'time_per_item': 16.84}
{'num': 11, 'card': 'P-1149', 'error': 0, 'tokens': 188, 'time_per_item': 25.41}
{'num': 12, 'card': 'P-1106', 'error': 0, 'tokens': 218, 'time_per_item': 28.32}
{'num': 13, 'card': 'P-1054', 'error': 0, 